# Notebook 02: VectorDB Comparison Experiments (Google Colab Version)

## MedQA Thesis — Systematic Comparison of RAG Architectures

> **This is the Google Colab version.** It includes Drive mounting and pip installs.  
> For the local version, see `notebooks/local/02_component_experiments.ipynb`

### Experiment Design

| Component | Value | Status |
|-----------|-------|--------|
| **Chunking** | Semantic Passage (200 tokens) | FIXED |
| **Embedding** | all-MiniLM-L6-v2 (384 dims) | FIXED |
| **LLM** | LLaMA 3.3 70B via Groq | FIXED |
| **VectorDB** | FAISS, ChromaDB, Qdrant, Pinecone | **VARIABLE** |
| **RAG Architectures** | Naive, Sparse, Hybrid, Multi-Hop | All tested per VectorDB |
| **Sample Size** | 5 dev set questions | - |

## 0. Google Colab Setup

Mount Google Drive and install all required dependencies.

In [ ]:
# ------------------------------------------------------------------
# GOOGLE COLAB SETUP
# ------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

# Project root on Google Drive
COLAB_PROJECT_ROOT = '/content/drive/MyDrive/MedQA-Thesis'

# Install all dependencies (suppress Colab's pre-installed package warnings)
import subprocess, sys

packages = [
    "langchain==0.3.25", "langchain-core==0.3.83", "langchain-community==0.3.25",
    "langchain-text-splitters==0.3.11", "langchain-openai==0.3.25",
    "sentence-transformers", "faiss-cpu", "rank-bm25", "groq", "chromadb",
    "qdrant-client", "pinecone",
    "openpyxl", "rouge-score", "bert-score",
    "pandas", "numpy", "matplotlib", "seaborn", "tqdm",
    "requests==2.32.4",
]

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q"] + packages,
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
print("All packages installed.")
print(f"Project root: {COLAB_PROJECT_ROOT}")

# ------------------------------------------------------------------
# Verify preprocessed data from Notebook 01 exists on Google Drive
# ------------------------------------------------------------------
import os
processed_path = os.path.join(COLAB_PROJECT_ROOT, 'data', 'processed')
excel_path = os.path.join(COLAB_PROJECT_ROOT, 'experiments', 'ThesisExperiment.xlsx')

expected_files = ["medqa_dev.parquet", "textbook_corpus.json"]
missing = [f for f in expected_files if not os.path.exists(os.path.join(processed_path, f))]

if missing:
    print(f"\nPreprocessed data NOT FOUND on Google Drive!")
    print(f"   Expected path: {processed_path}")
    print(f"   Missing files: {missing}")
    print("\n   Please run Notebook 01 first to generate these files,")
    print("   or upload them to: My Drive / MedQA-Thesis / data / processed /")
    raise FileNotFoundError(f"Missing preprocessed data: {missing}")
else:
    print(f"\nPreprocessed data found on Google Drive:")
    for f in expected_files:
        fpath = os.path.join(processed_path, f)
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"  {f}: {size_mb:.2f} MB")

if os.path.exists(excel_path):
    print(f"Excel file found: {excel_path}")
else:
    print(f"Excel file not found at {excel_path}")
    print("   Metrics will not be written to Excel until the file is uploaded.")

## 1. Setup & Imports

In [ ]:
# Packages already installed in Colab Setup cell above

# ------------------------------------------------------------------
# Standard library
# ------------------------------------------------------------------
import json
import os
import re
import time
import pickle
import warnings
from pathlib import Path
from collections import Counter

# ------------------------------------------------------------------
# Data & Visualisation
# ------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import openpyxl

# ------------------------------------------------------------------
# NLP / ML
# ------------------------------------------------------------------
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
import faiss
from rank_bm25 import BM25Okapi
import chromadb
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

# ------------------------------------------------------------------
# LLM
# ------------------------------------------------------------------
from groq import Groq

# ------------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------------
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn

warnings.filterwarnings("ignore")

# ------------------------------------------------------------------
# Project paths
# ------------------------------------------------------------------
PROJECT_ROOT   = Path(COLAB_PROJECT_ROOT)
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_INDICES   = PROJECT_ROOT / "data" / "indices"
EXPERIMENTS    = PROJECT_ROOT / "experiments"
RESULTS_DIR    = PROJECT_ROOT / "results" / "experiments"
EXCEL_PATH     = EXPERIMENTS / "ThesisExperiment.xlsx"

# Create output directories
for d in [DATA_INDICES, RESULTS_DIR, RESULTS_DIR / "plots"]:
    d.mkdir(parents=True, exist_ok=True)

print("Setup complete. Project root:", PROJECT_ROOT.resolve())

In [ ]:
# ------------------------------------------------------------------
# API KEY CONFIGURATION
# Option 1: Direct (for quick testing)
GROQ_API_KEY = "your-groq-api-key-here"
PINECONE_API_KEY = "your-pinecone-api-key-here"  # Leave as-is to skip Pinecone

# Option 2: From config file on Drive
# sys.path.insert(0, str(PROJECT_ROOT / "config"))
# from config import GROQ_API_KEY

# Initialise Groq client
groq_client = Groq(api_key=GROQ_API_KEY)
print("Groq client initialised successfully.")

# Check Pinecone availability
USE_PINECONE = PINECONE_API_KEY != "your-pinecone-api-key-here"
print(f"Pinecone: {'ready' if USE_PINECONE else 'skipped (no API key)'}")

## 2. Load Preprocessed Data

We load the cleaned dev-set questions and textbook corpus from Notebook 01.  
A random sample of **30 questions** is drawn for these component experiments.

In [ ]:
# Load dev set questions
df_dev = pd.read_parquet(DATA_PROCESSED / "medqa_dev.parquet")
print(f"Dev set: {len(df_dev)} questions")
print(f"Columns: {list(df_dev.columns)}")

# Reconstruct 'options' dict column from individual option columns
df_dev["options"] = df_dev.apply(
    lambda row: {"A": row["option_a"], "B": row["option_b"], 
                 "C": row["option_c"], "D": row["option_d"]}, axis=1
)
print("Reconstructed 'options' column as dict")

# Load textbook corpus
with open(DATA_PROCESSED / "textbook_corpus.json", "r") as f:
    textbook_corpus = json.load(f)
print(f"Textbook corpus: {len(textbook_corpus)} books")

# Sample 5 questions (reproducible)
SAMPLE_SIZE = 5
df_sample = df_dev.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
print(f"\nSampled {SAMPLE_SIZE} questions for experiments.")
df_sample.head()

## 3. Fixed Experiment Configuration

All experiments share these constants. Only the **VectorDB** and **RAG architecture** change.

In [ ]:
# ==============================================================
# FIXED CONFIGURATION - DO NOT CHANGE DURING EXPERIMENTS
# ==============================================================
CHUNKING_NAME       = "Semantic Passage (200 tokens)"
CHUNK_MAX_TOKENS    = 200          # ~800 characters
CHUNK_OVERLAP       = 20           # small overlap in characters

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
EMBEDDING_DIM        = 384

LLM_MODEL           = "llama-3.3-70b-versatile"
LLM_TEMPERATURE     = 0           # deterministic output
TOP_K               = 5           # retrieve top-5 chunks
MAX_HOPS            = 3           # for Multi-Hop RAG
RRF_K               = 60          # for Hybrid RAG fusion

FAITHFULNESS_THRESHOLD = 0.5      # below this -> flagged as hallucination

# Excel row mapping: (rag_type, vectordb) -> Excel row number
# Rows 4/5, 9/10, 14/15, 19/20 are for Qdrant and Pinecone
EXCEL_ROW_MAP = {
    ("naive",    "FAISS"):    2,  ("naive",    "ChromaDB"): 3,
    ("naive",    "Qdrant"):   4,  ("naive",    "Pinecone"): 5,
    ("sparse",   "FAISS"):    7,  ("sparse",   "ChromaDB"): 8,
    ("sparse",   "Qdrant"):   9,  ("sparse",   "Pinecone"): 10,
    ("hybrid",   "FAISS"):    12, ("hybrid",   "ChromaDB"): 13,
    ("hybrid",   "Qdrant"):   14, ("hybrid",   "Pinecone"): 15,
    ("multihop", "FAISS"):    17, ("multihop", "ChromaDB"): 18,
    ("multihop", "Qdrant"):   19, ("multihop", "Pinecone"): 20,
}

print("Fixed configuration loaded.")
print(f"  Chunking:  {CHUNKING_NAME}")
print(f"  Embedding: {EMBEDDING_MODEL_NAME} ({EMBEDDING_DIM}d)")
print(f"  LLM:       {LLM_MODEL}")
print(f"  Top-K:     {TOP_K}")
print(f"  Sample:    {SAMPLE_SIZE} questions")

## 4. Text Chunking — Semantic Passage (200 tokens)

Semantic Passage chunking splits text at natural boundaries (paragraphs, sections) while respecting a maximum token limit. This preserves complete medical concepts within each chunk.

**Strategy**: Use `RecursiveCharacterTextSplitter` with paragraph-level separators and a word-count approximation for the 200-token limit (~800 characters).

In [ ]:
def semantic_passage_chunk(text, max_tokens=200, overlap=20):
    """
    Semantic Passage chunking: splits at natural text boundaries
    (triple newlines, double newlines, single newlines, sentences)
    while keeping each chunk under max_tokens words.
    
    Parameters
    ----------
    text : str — The full text to chunk
    max_tokens : int — Maximum words per chunk (~tokens)
    overlap : int — Character overlap between chunks
    
    Returns
    -------
    list[str] — List of text chunks
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=max_tokens * 4,        # ~4 chars per token approximation
        chunk_overlap=overlap,
        separators=["\n\n\n", "\n\n", "\n", ". ", "; ", ", ", " "],
        length_function=lambda x: len(x.split()),  # word-count proxy for tokens
        is_separator_regex=False,
    )
    chunks = splitter.split_text(text)
    # Filter out very short chunks (< 10 words)
    chunks = [c.strip() for c in chunks if len(c.split()) >= 10]
    return chunks


# ------------------------------------------------------------------
# Chunk the entire textbook corpus
# ------------------------------------------------------------------
print("Chunking textbook corpus with Semantic Passage strategy...")
all_chunks = []         # list of chunk texts
chunk_metadata = []     # list of {book_name, chunk_id, text}

for book in tqdm(textbook_corpus, desc="Chunking books"):
    book_name = book["book_name"]
    book_text = book["text"]
    book_chunks = semantic_passage_chunk(book_text, max_tokens=CHUNK_MAX_TOKENS, overlap=CHUNK_OVERLAP)
    
    for i, chunk_text in enumerate(book_chunks):
        all_chunks.append(chunk_text)
        chunk_metadata.append({
            "book_name": book_name,
            "chunk_id": f"{book_name}_chunk_{i}",
            "text": chunk_text,
        })

print(f"\nTotal chunks: {len(all_chunks):,}")
print(f"Avg chunk length: {np.mean([len(c.split()) for c in all_chunks]):.1f} words")
print(f"Min: {min(len(c.split()) for c in all_chunks)} words | Max: {max(len(c.split()) for c in all_chunks)} words")

## 5. Embedding Generation

We embed all chunks using **all-MiniLM-L6-v2** (384 dimensions, 22M parameters).  
Embeddings are normalised for cosine similarity via inner product search.

In [ ]:
# Load embedding model
print(f"Loading embedding model: {EMBEDDING_MODEL_NAME}")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print(f"Model loaded. Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")

# Embed all chunks (batch processing)
print(f"\nEmbedding {len(all_chunks):,} chunks...")
chunk_embeddings = embedding_model.encode(
    all_chunks,
    show_progress_bar=True,
    batch_size=256,
    normalize_embeddings=True,   # L2-normalise for cosine similarity via inner product
)
chunk_embeddings = np.array(chunk_embeddings, dtype=np.float32)
print(f"Embeddings shape: {chunk_embeddings.shape}")

## 6. Build BM25 Index (Shared)

BM25 is used by **Sparse RAG** and **Hybrid RAG**. It is keyword-based and independent of the VectorDB choice, so we build it once.

In [ ]:
# Tokenise chunks for BM25
print("Building BM25 index...")
tokenised_chunks = [chunk.lower().split() for chunk in all_chunks]
bm25_index = BM25Okapi(tokenised_chunks)
print(f"BM25 index built over {len(tokenised_chunks):,} chunks.")

## 7. Evaluation Metrics

We compute **20 metrics** matching the Excel headers:

| Category | Metrics |
|----------|---------|
| Classification (MCQ) | Accuracy, Precision, Recall, F1 |
| Token-level | Token_F1, Exact Match |
| Text quality | ROUGE-1, ROUGE-L |
| Semantic similarity | BERTScore (Precision, Recall, F1) |
| Retrieval quality | Recall@3, Recall@5, Recall@10, MRR |
| RAGAS (LLM-based) | Faithfulness, Hallucination Rate, Answer Relevance, Context Precision, Context Recall |

In [ ]:
# ==============================================================
# ANSWER PARSING
# ==============================================================
def parse_answer_letter(response_text):
    """Extract the answer letter (A/B/C/D) from LLM response."""
    if not response_text:
        return None
    
    # Pattern 1: "Answer: A" or "Answer: B)"
    match = re.search(r"[Aa]nswer\s*[:=]\s*\(?([A-Da-d])\)?", response_text)
    if match:
        return match.group(1).upper()
    
    # Pattern 2: standalone letter at start of line
    match = re.search(r"^\s*\(?([A-Da-d])\)?[\s\)\.]", response_text, re.MULTILINE)
    if match:
        return match.group(1).upper()
    
    # Pattern 3: first letter A-D mentioned
    match = re.search(r"\b([A-Da-d])\b", response_text)
    if match:
        return match.group(1).upper()
    
    return None


# ==============================================================
# CLASSIFICATION METRICS (MCQ: A/B/C/D)
# ==============================================================
def compute_classification_metrics(predictions, ground_truths):
    """
    Compute Accuracy, Precision, Recall, F1 for MCQ predictions.
    Treats each option (A/B/C/D) as a class; macro-averages P/R/F1.
    
    Returns: dict with Accuracy, Precision, Recall, F1
    """
    correct = sum(1 for p, g in zip(predictions, ground_truths) if p == g)
    accuracy = correct / len(predictions) if predictions else 0
    
    # Per-class precision/recall for macro averaging
    labels = ["A", "B", "C", "D"]
    precisions, recalls = [], []
    
    for label in labels:
        tp = sum(1 for p, g in zip(predictions, ground_truths) if p == label and g == label)
        fp = sum(1 for p, g in zip(predictions, ground_truths) if p == label and g != label)
        fn = sum(1 for p, g in zip(predictions, ground_truths) if p != label and g == label)
        
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0
        precisions.append(prec)
        recalls.append(rec)
    
    macro_precision = np.mean(precisions)
    macro_recall    = np.mean(recalls)
    macro_f1 = (2 * macro_precision * macro_recall / (macro_precision + macro_recall)
                if (macro_precision + macro_recall) > 0 else 0)
    
    return {
        "Accuracy":  round(accuracy, 4),
        "Precision": round(macro_precision, 4),
        "Recall":    round(macro_recall, 4),
        "F1":        round(macro_f1, 4),
    }


# ==============================================================
# TOKEN-LEVEL METRICS
# ==============================================================
def compute_token_f1(prediction_text, reference_text):
    """Token-level F1 between two strings."""
    pred_tokens = prediction_text.lower().split()
    ref_tokens  = reference_text.lower().split()
    common = Counter(pred_tokens) & Counter(ref_tokens)
    num_common = sum(common.values())
    
    if num_common == 0:
        return 0.0
    prec = num_common / len(pred_tokens) if pred_tokens else 0
    rec  = num_common / len(ref_tokens) if ref_tokens else 0
    return 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0


def compute_exact_match(prediction, reference):
    """Binary exact match (case-insensitive)."""
    return 1.0 if prediction.strip().upper() == reference.strip().upper() else 0.0


# ==============================================================
# ROUGE METRICS
# ==============================================================
rouge_scorer_obj = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)

def compute_rouge(prediction_text, reference_text):
    """Compute ROUGE-1 and ROUGE-L F1 scores."""
    scores = rouge_scorer_obj.score(reference_text, prediction_text)
    return {
        "ROUGE_1": round(scores["rouge1"].fmeasure, 4),
        "ROUGE_L": round(scores["rougeL"].fmeasure, 4),
    }


# ==============================================================
# RETRIEVAL METRICS
# ==============================================================
def compute_retrieval_metrics(retrieved_chunks_list, df_questions, all_chunk_texts):
    """
    Compute Recall@k and MRR.
    A retrieved chunk is 'relevant' if it contains any key term from the correct answer.
    
    Parameters
    ----------
    retrieved_chunks_list : list[list[int]] — per-question list of retrieved chunk indices
    df_questions : DataFrame — must have 'answer' or 'answer_idx' and 'options' columns
    all_chunk_texts : list[str] — full chunk text corpus
    
    Returns: dict with Recall@3, Recall@5, Recall@10, MRR
    """
    recall_at = {3: [], 5: [], 10: []}
    mrr_scores = []
    
    for i, indices in enumerate(retrieved_chunks_list):
        row = df_questions.iloc[i]
        # Get the correct answer text
        if "answer" in row and isinstance(row["answer"], str):
            answer_text = row["answer"].lower()
        elif "answer_idx" in row and "options" in row:
            opts = row["options"]
            idx_map = {"A": 0, "B": 1, "C": 2, "D": 3}
            answer_text = list(opts.values())[idx_map.get(row["answer_idx"], 0)].lower()
        else:
            answer_text = ""
        
        # Extract key terms from the answer (words > 3 chars)
        key_terms = [w for w in answer_text.split() if len(w) > 3]
        if not key_terms:
            key_terms = answer_text.split()
        
        # Check relevance of each retrieved chunk
        found_rank = None
        for rank, idx in enumerate(indices):
            if idx < len(all_chunk_texts):
                chunk_lower = all_chunk_texts[idx].lower()
                # A chunk is relevant if it contains at least one key term
                if any(term in chunk_lower for term in key_terms):
                    if found_rank is None:
                        found_rank = rank + 1  # 1-indexed
        
        for k in [3, 5, 10]:
            top_k_indices = indices[:k]
            hit = 0
            for idx in top_k_indices:
                if idx < len(all_chunk_texts):
                    chunk_lower = all_chunk_texts[idx].lower()
                    if any(term in chunk_lower for term in key_terms):
                        hit = 1
                        break
            recall_at[k].append(hit)
        
        mrr_scores.append(1.0 / found_rank if found_rank else 0.0)
    
    return {
        "Recall@3":  round(np.mean(recall_at[3]), 4),
        "Recall@5":  round(np.mean(recall_at[5]), 4),
        "Recall@10": round(np.mean(recall_at[10]), 4),
        "MRR":       round(np.mean(mrr_scores), 4),
    }


# ==============================================================
# RAGAS METRICS (using Groq LLM as evaluator)
# ==============================================================
def ragas_faithfulness_single(answer_text, context_text, groq_client, model=LLM_MODEL):
    """
    Compute faithfulness for a single answer: what fraction of claims
    in the answer are supported by the context?
    Uses Groq LLM to extract claims and verify each.
    Returns a score between 0 and 1.
    """
    if not answer_text or not context_text:
        return 0.0
    
    # Step 1: Extract claims from the answer
    extract_prompt = f"""Extract all factual claims from the following answer. 
Return each claim on a separate line, numbered. If no clear claims, return "NO_CLAIMS".

Answer: {answer_text}

Claims:"""
    
    try:
        resp = groq_client.chat.completions.create(
            model=model, temperature=0, max_tokens=500,
            messages=[{"role": "user", "content": extract_prompt}]
        )
        claims_text = resp.choices[0].message.content.strip()
        
        if "NO_CLAIMS" in claims_text:
            return 1.0
        
        # Parse claims
        claims = [line.strip() for line in claims_text.split("\n") if line.strip() and line.strip()[0].isdigit()]
        if not claims:
            return 1.0
        
        # Step 2: Verify each claim against context
        supported = 0
        for claim in claims:
            verify_prompt = f"""Given the following context, determine if this claim is supported.
Answer only "SUPPORTED" or "NOT_SUPPORTED".

Context: {context_text[:3000]}

Claim: {claim}

Verdict:"""
            resp2 = groq_client.chat.completions.create(
                model=model, temperature=0, max_tokens=20,
                messages=[{"role": "user", "content": verify_prompt}]
            )
            verdict = resp2.choices[0].message.content.strip().upper()
            if "SUPPORTED" in verdict and "NOT" not in verdict:
                supported += 1
            time.sleep(0.5)  # rate limiting
        
        return supported / len(claims)
    except Exception as e:
        print(f"    Faithfulness error: {e}")
        return np.nan


def ragas_answer_relevance_single(question, answer_text, groq_client, model=LLM_MODEL):
    """Score how relevant the answer is to the question (0-1)."""
    if not answer_text:
        return 0.0
    
    prompt = f"""Rate how relevant this answer is to the question on a scale of 0.0 to 1.0.
Return ONLY a decimal number.

Question: {question}
Answer: {answer_text}

Relevance score:"""
    try:
        resp = groq_client.chat.completions.create(
            model=model, temperature=0, max_tokens=10,
            messages=[{"role": "user", "content": prompt}]
        )
        score_text = resp.choices[0].message.content.strip()
        score = float(re.search(r"[0-9]*\.?[0-9]+", score_text).group())
        return min(max(score, 0.0), 1.0)
    except:
        return np.nan


def ragas_context_precision_single(question, context_chunks, answer_text, groq_client, model=LLM_MODEL):
    """Evaluate if relevant chunks are ranked higher than irrelevant ones."""
    if not context_chunks:
        return 0.0
    
    prompt = f"""For each context passage below, determine if it is relevant to answering the question.
Return a comma-separated list of 1 (relevant) or 0 (not relevant) for each passage IN ORDER.

Question: {question}

{chr(10).join(f"Passage {i+1}: {c[:300]}" for i, c in enumerate(context_chunks[:5]))}

Relevance (comma-separated):"""
    try:
        resp = groq_client.chat.completions.create(
            model=model, temperature=0, max_tokens=50,
            messages=[{"role": "user", "content": prompt}]
        )
        nums = re.findall(r"[01]", resp.choices[0].message.content)
        relevance = [int(n) for n in nums[:len(context_chunks)]]
        
        if not any(relevance):
            return 0.0
        
        # Average precision: (1/K) * sum(Precision@k * Rel_k)
        precisions = []
        running_relevant = 0
        for k, rel in enumerate(relevance):
            if rel:
                running_relevant += 1
                precisions.append(running_relevant / (k + 1))
        
        return np.mean(precisions) if precisions else 0.0
    except:
        return np.nan


def ragas_context_recall_single(question, context_text, answer_text, groq_client, model=LLM_MODEL):
    """Evaluate if context contains all information needed for the correct answer."""
    if not context_text:
        return 0.0
    
    prompt = f"""Given the question and correct answer below, does the context contain sufficient 
information to derive the correct answer? Rate from 0.0 to 1.0.
Return ONLY a decimal number.

Question: {question}
Correct Answer: {answer_text}
Context: {context_text[:3000]}

Context recall score:"""
    try:
        resp = groq_client.chat.completions.create(
            model=model, temperature=0, max_tokens=10,
            messages=[{"role": "user", "content": prompt}]
        )
        score_text = resp.choices[0].message.content.strip()
        score = float(re.search(r"[0-9]*\.?[0-9]+", score_text).group())
        return min(max(score, 0.0), 1.0)
    except:
        return np.nan


print("All evaluation metric functions defined.")

## 8. RAG Pipeline Components

### Shared components across all architectures:
- **Prompt template**: Evidence-grounded, instructs LLM to answer ONLY from retrieved passages
- **LLM call**: Groq API with temperature=0 for reproducibility
- **Answer parser**: Extracts A/B/C/D from LLM response

In [ ]:
# ==============================================================
# PROMPT TEMPLATE (shared across ALL RAG architectures)
# ==============================================================
PROMPT_TEMPLATE = """You are a medical expert answering USMLE-style clinical questions.
Use ONLY the provided evidence passages to answer. If the evidence
is insufficient, state that rather than guessing.

Evidence Passages:
{context}

Question: {question}

Options:
A) {option_a}
B) {option_b}
C) {option_c}
D) {option_d}

Instructions:
1. Analyze the evidence passages relevant to the question.
2. Select the single best answer (A, B, C, or D).
3. Provide a brief explanation citing specific evidence passages.

Answer format:
Answer: [letter]
Explanation: [your reasoning with evidence citations]
"""

# Maximum characters per context chunk to control token usage
# ~4 chars per token, 300 tokens per chunk × 5 chunks = ~1500 tokens for context
MAX_CHUNK_CHARS = 1200


def build_prompt(question, options, context_chunks):
    """Build the prompt from question, options and retrieved context."""
    # Truncate each chunk to limit token usage
    truncated = [c[:MAX_CHUNK_CHARS] for c in context_chunks]
    context_str = "\n\n".join(
        f"[Passage {i+1}]: {chunk}" for i, chunk in enumerate(truncated)
    )
    # Handle options as dict or list
    if isinstance(options, dict):
        opt_values = list(options.values())
    else:
        opt_values = options
    
    return PROMPT_TEMPLATE.format(
        context=context_str,
        question=question,
        option_a=opt_values[0] if len(opt_values) > 0 else "",
        option_b=opt_values[1] if len(opt_values) > 1 else "",
        option_c=opt_values[2] if len(opt_values) > 2 else "",
        option_d=opt_values[3] if len(opt_values) > 3 else "",
    )


def call_llm(prompt, groq_client, model=LLM_MODEL, temperature=0, max_tokens=500):
    """Call Groq LLM and return the response text."""
    try:
        response = groq_client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature,
            max_tokens=max_tokens,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"  LLM Error: {e}")
        time.sleep(2)
        return ""


# ==============================================================
# RETRIEVAL FUNCTIONS
# ==============================================================
def faiss_retrieve(query_embedding, faiss_index, top_k=5):
    """Retrieve top-k chunks from FAISS index."""
    query_emb = np.array([query_embedding], dtype=np.float32)
    scores, indices = faiss_index.search(query_emb, top_k)
    return indices[0].tolist(), scores[0].tolist()


def chromadb_retrieve(query_embedding, chroma_collection, top_k=5):
    """Retrieve top-k chunks from ChromaDB collection."""
    results = chroma_collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k,
    )
    ids = results["ids"][0]
    indices = [int(id_str) for id_str in ids]
    distances = results["distances"][0] if results["distances"] else [0] * len(ids)
    return indices, distances


def bm25_retrieve(query_text, bm25_index, top_k=5):
    """Retrieve top-k chunks using BM25 keyword search."""
    query_tokens = query_text.lower().split()
    scores = bm25_index.get_scores(query_tokens)
    top_indices = np.argsort(scores)[::-1][:top_k]
    top_scores = scores[top_indices]
    return top_indices.tolist(), top_scores.tolist()


def rrf_fuse(ranking_lists, k=60):
    """
    Reciprocal Rank Fusion: merge multiple ranked lists.
    RRF(d) = sum over rankings of 1/(k + rank(d))
    """
    fused_scores = {}
    for ranking in ranking_lists:
        for rank, (idx, _) in enumerate(ranking):
            if idx not in fused_scores:
                fused_scores[idx] = 0.0
            fused_scores[idx] += 1.0 / (k + rank + 1)
    
    sorted_results = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_results


print("RAG pipeline components defined.")

# ==============================================================
# ADDITIONAL RETRIEVAL FUNCTIONS (Qdrant, Pinecone)
# ==============================================================
def qdrant_retrieve(query_emb, qdrant_client_obj, collection_name, top_k=5):
    """Retrieve top-k chunks from Qdrant collection."""
    results = qdrant_client_obj.query_points(
        collection_name=collection_name,
        query=query_emb.tolist(),
        limit=top_k,
    )
    return [hit.id for hit in results.points], [hit.score for hit in results.points]


def pinecone_retrieve(query_emb, pinecone_idx, top_k=5):
    """Retrieve top-k chunks from Pinecone index."""
    results = pinecone_idx.query(vector=query_emb.tolist(), top_k=top_k, include_metadata=False)
    return [int(m["id"]) for m in results["matches"]], [m["score"] for m in results["matches"]]


def dense_retrieve(query_emb, vectordb_type, top_k=5, faiss_idx=None,
                   chroma_col=None, qdrant_cl=None, qdrant_col_name=None, pinecone_idx=None):
    """Generic dispatcher: route retrieval to the correct VectorDB."""
    if vectordb_type == "FAISS":
        return faiss_retrieve(query_emb, faiss_idx, top_k)
    elif vectordb_type == "ChromaDB":
        return chromadb_retrieve(query_emb, chroma_col, top_k)
    elif vectordb_type == "Qdrant":
        return qdrant_retrieve(query_emb, qdrant_cl, qdrant_col_name, top_k)
    elif vectordb_type == "Pinecone":
        return pinecone_retrieve(query_emb, pinecone_idx, top_k)
    else:
        raise ValueError(f"Unknown VectorDB: {vectordb_type}")

print("All retrieval functions defined (FAISS, ChromaDB, Qdrant, Pinecone).")

## 9. RAG Architecture Implementations

Each architecture takes the same inputs and returns the same output format:
- **Input**: question, options, retrieval index/indices
- **Output**: predicted letter, full response text, retrieved chunk indices, retrieved chunk texts

In [ ]:
# ==============================================================
# NAIVE RAG: Query -> Dense Retrieval -> Generate
# ==============================================================
def run_naive_rag(question, options, embedding_model, vectordb_type, top_k=5, **vdb_kwargs):
    """Naive RAG: embed query -> dense retrieval -> prompt -> LLM -> answer"""
    query_emb = embedding_model.encode(question, normalize_embeddings=True)
    indices, scores = dense_retrieve(query_emb, vectordb_type, top_k, **vdb_kwargs)
    context_chunks = [all_chunks[i] for i in indices if i < len(all_chunks)]
    prompt = build_prompt(question, options, context_chunks)
    response = call_llm(prompt, groq_client)
    predicted = parse_answer_letter(response)
    return predicted, response, indices, context_chunks


# ==============================================================
# SPARSE RAG: Query -> BM25 Keyword Retrieval -> Generate
# ==============================================================
def run_sparse_rag(question, options, bm25_idx, top_k=5):
    """Sparse RAG: BM25 keyword search -> prompt -> LLM -> answer.
    Note: BM25 is independent of VectorDB choice."""
    indices, scores = bm25_retrieve(question, bm25_idx, top_k)
    context_chunks = [all_chunks[i] for i in indices if i < len(all_chunks)]
    prompt = build_prompt(question, options, context_chunks)
    response = call_llm(prompt, groq_client)
    predicted = parse_answer_letter(response)
    return predicted, response, indices, context_chunks


# ==============================================================
# HYBRID RAG: (Dense || BM25) -> RRF Fusion -> Generate
# ==============================================================
def run_hybrid_rag(question, options, embedding_model, bm25_idx,
                   vectordb_type, top_k=5, rrf_k=60, **vdb_kwargs):
    """Hybrid RAG: parallel dense + BM25 -> RRF fusion -> prompt -> LLM -> answer"""
    query_emb = embedding_model.encode(question, normalize_embeddings=True)
    fetch_k = top_k * 3
    dense_indices, dense_scores = dense_retrieve(query_emb, vectordb_type, fetch_k, **vdb_kwargs)
    sparse_indices, sparse_scores = bm25_retrieve(question, bm25_idx, fetch_k)
    dense_ranking = list(zip(dense_indices, dense_scores))
    sparse_ranking = list(zip(sparse_indices, sparse_scores))
    fused = rrf_fuse([dense_ranking, sparse_ranking], k=rrf_k)
    top_indices = [idx for idx, _ in fused[:top_k]]
    context_chunks = [all_chunks[i] for i in top_indices if i < len(all_chunks)]
    prompt = build_prompt(question, options, context_chunks)
    response = call_llm(prompt, groq_client)
    predicted = parse_answer_letter(response)
    return predicted, response, top_indices, context_chunks


# ==============================================================
# MULTI-HOP RAG: Decompose -> Iterative Retrieval -> Generate
# ==============================================================
def run_multihop_rag(question, options, embedding_model, vectordb_type,
                     top_k=5, max_hops=3, **vdb_kwargs):
    """Multi-Hop RAG: decompose question -> iterative retrieval (up to max_hops) -> answer"""
    reasoning_chain = []
    all_retrieved_indices = []
    all_context_chunks = []

    # Step 1: Question Decomposition
    decompose_prompt = f"""Break down this medical question into 2-3 focused sub-questions,
each targeting a specific piece of clinical information needed.
Return each sub-question on a separate numbered line.

Question: {question}

Sub-questions:"""
    decompose_resp = call_llm(decompose_prompt, groq_client, max_tokens=200)
    sub_questions = [
        line.strip() for line in decompose_resp.split("\n")
        if line.strip() and (line.strip()[0].isdigit() or line.strip().startswith("-"))
    ]
    if not sub_questions:
        sub_questions = [question]
    reasoning_chain.append({"step": "decomposition", "sub_questions": sub_questions})

    # Hop 1: Initial retrieval for each sub-question
    for sq in sub_questions:
        query_emb = embedding_model.encode(sq, normalize_embeddings=True)
        indices, scores = dense_retrieve(query_emb, vectordb_type, top_k, **vdb_kwargs)
        chunks = [all_chunks[i] for i in indices if i < len(all_chunks)]
        all_retrieved_indices.extend(indices)
        all_context_chunks.extend(chunks)
    reasoning_chain.append({"step": "hop_1", "num_chunks": len(all_context_chunks)})

    # Hop 2+: Iterative refinement
    for hop in range(2, max_hops + 1):
        accumulated_evidence = "\n".join(all_context_chunks[-10:])
        gap_prompt = f"""Given the question and evidence gathered so far, identify any missing information.
If all information is sufficient, respond with "SUFFICIENT".
Otherwise, generate 1-2 refined search queries to fill the gaps.

Question: {question}
Evidence so far: {accumulated_evidence[:2000]}

Assessment:"""
        gap_resp = call_llm(gap_prompt, groq_client, max_tokens=200)
        time.sleep(0.5)
        if "SUFFICIENT" in gap_resp.upper():
            reasoning_chain.append({"step": f"hop_{hop}", "status": "sufficient"})
            break
        refined_queries = [
            line.strip() for line in gap_resp.split("\n")
            if line.strip() and len(line.strip()) > 10
        ][:2]
        if not refined_queries:
            break
        for rq in refined_queries:
            query_emb = embedding_model.encode(rq, normalize_embeddings=True)
            indices, scores = dense_retrieve(query_emb, vectordb_type, top_k, **vdb_kwargs)
            chunks = [all_chunks[i] for i in indices if i < len(all_chunks)]
            all_retrieved_indices.extend(indices)
            all_context_chunks.extend(chunks)
        reasoning_chain.append({"step": f"hop_{hop}", "refined_queries": refined_queries})

    # Deduplicate evidence
    seen = set()
    unique_indices, unique_chunks = [], []
    for idx, chunk in zip(all_retrieved_indices, all_context_chunks):
        if idx not in seen:
            seen.add(idx)
            unique_indices.append(idx)
            unique_chunks.append(chunk)
    final_indices = unique_indices[:top_k * 2]
    final_chunks = unique_chunks[:top_k * 2]

    # Final answer generation
    prompt = build_prompt(question, options, final_chunks)
    response = call_llm(prompt, groq_client)
    predicted = parse_answer_letter(response)
    return predicted, response, final_indices, final_chunks, reasoning_chain


print("All 4 RAG architectures defined: Naive, Sparse, Hybrid, Multi-Hop")

## 10. Excel Writer & Experiment Runner

The Excel writer updates `experiments/ThesisExperiment.xlsx` after each experiment.  
It maps metrics to the correct columns (G through Z) at the experiment's row.

In [ ]:
# ==============================================================
# EXCEL COLUMN MAPPING (matches ThesisExperiment.xlsx headers)
# ==============================================================
EXCEL_COL_MAP = {
    "A": "SN", "B": "RAG_Name", "C": "Chunking_Technique", "D": "Embedding",
    "E": "VectorDB", "F": "Generator_Model",
    "G": "Accuracy", "H": "Precision", "I": "Recall", "J": "F1",
    "K": "Token_F1", "L": "Exact_Match",
    "M": "ROUGE_1", "N": "ROUGE_L",
    "O": "BERTScore_P", "P": "BERTScore_R", "Q": "BERTScore_F1",
    "R": "Recall@3", "S": "Recall@5", "T": "Recall@10", "U": "MRR",
    "V": "RAGAS_Faithfulness", "W": "RAGAS_Hallucination_Rate",
    "X": "RAGAS_Answer_Relevance", "Y": "RAGAS_Context_Precision", "Z": "RAGAS_Context_Recall",
}
METRIC_TO_COL = {v: k for k, v in EXCEL_COL_MAP.items()}


def write_experiment_to_excel(excel_path, row_num, config_dict, metrics_dict):
    """Write experiment config and metrics to Excel at the specified row."""
    if not excel_path.exists():
        print(f"  Excel not found, skipping write"); return
    wb = openpyxl.load_workbook(excel_path)
    ws = wb["Vector DB comparision"]
    for key, value in {**config_dict, **metrics_dict}.items():
        col = METRIC_TO_COL.get(key)
        if col and value is not None:
            ws[f"{col}{row_num}"] = round(value, 4) if isinstance(value, float) else value
    wb.save(excel_path)
    print(f"  Excel updated: row {row_num} in {excel_path.name}")


# ==============================================================
# MASTER EXPERIMENT RUNNER
# ==============================================================
all_results = {}

def run_full_experiment(rag_type, vectordb_name, df_questions, **vdb_kwargs):
    """
    Run a complete experiment: retrieve -> generate -> evaluate -> write to Excel.
    
    Parameters
    ----------
    rag_type : str - "naive", "sparse", "hybrid", "multihop"
    vectordb_name : str - "FAISS", "ChromaDB", "Qdrant", "Pinecone"
    df_questions : DataFrame - sampled questions
    **vdb_kwargs : VectorDB-specific args (faiss_idx, chroma_col, qdrant_cl, etc.)
    """
    rag_display = {"naive": "Naive RAG", "sparse": "Sparse RAG",
                   "hybrid": "Hybrid RAG", "multihop": "Multi-Hop RAG"}[rag_type]
    excel_row = EXCEL_ROW_MAP.get((rag_type, vectordb_name))

    print(f"\n{'='*60}")
    print(f"EXPERIMENT: {rag_display} + {vectordb_name}")
    print(f"Excel Row: {excel_row} | Questions: {len(df_questions)}")
    print(f"{'='*60}")

    predictions, ground_truths, responses = [], [], []
    all_retrieved_indices, all_context_texts = [], []
    correct_answer_texts, questions_text = [], []

    for idx in tqdm(range(len(df_questions)), desc=f"{rag_display} + {vectordb_name}"):
        row = df_questions.iloc[idx]
        question = row["question"]
        options = row["options"]

        gt_letter = row.get("answer_idx", row.get("answer", "A"))
        if isinstance(options, dict):
            opt_values = list(options.values())
            idx_map = {"A": 0, "B": 1, "C": 2, "D": 3}
            correct_text = opt_values[idx_map.get(gt_letter, 0)]
        else:
            correct_text = str(options[{"A":0,"B":1,"C":2,"D":3}.get(gt_letter, 0)])

        # Run the appropriate RAG pipeline
        if rag_type == "naive":
            pred, resp, ret_idx, ret_chunks = run_naive_rag(
                question, options, embedding_model, vectordb_name, TOP_K, **vdb_kwargs)
        elif rag_type == "sparse":
            pred, resp, ret_idx, ret_chunks = run_sparse_rag(
                question, options, bm25_index, TOP_K)
        elif rag_type == "hybrid":
            pred, resp, ret_idx, ret_chunks = run_hybrid_rag(
                question, options, embedding_model, bm25_index,
                vectordb_name, TOP_K, RRF_K, **vdb_kwargs)
        elif rag_type == "multihop":
            pred, resp, ret_idx, ret_chunks, chain = run_multihop_rag(
                question, options, embedding_model, vectordb_name,
                TOP_K, MAX_HOPS, **vdb_kwargs)

        predictions.append(pred if pred else "X")
        ground_truths.append(gt_letter)
        responses.append(resp)
        all_retrieved_indices.append(ret_idx)
        all_context_texts.append(ret_chunks)
        correct_answer_texts.append(correct_text)
        questions_text.append(question)
        time.sleep(0.5)

    # ==============================================================
    # COMPUTE ALL METRICS
    # ==============================================================
    print("\nComputing metrics...")

    # 1. Classification metrics
    cls_metrics = compute_classification_metrics(predictions, ground_truths)
    print(f"  Accuracy: {cls_metrics['Accuracy']:.4f}")

    # 2. Token-level metrics
    avg_token_f1 = round(np.mean([compute_token_f1(p, g) for p, g in zip(predictions, ground_truths)]), 4)
    avg_exact_match = round(np.mean([compute_exact_match(p, g) for p, g in zip(predictions, ground_truths)]), 4)

    # 3. ROUGE metrics
    rouge1_scores, rougeL_scores = [], []
    for resp, correct in zip(responses, correct_answer_texts):
        if resp and correct:
            r = compute_rouge(resp, correct)
            rouge1_scores.append(r["ROUGE_1"]); rougeL_scores.append(r["ROUGE_L"])
        else:
            rouge1_scores.append(0.0); rougeL_scores.append(0.0)

    # 4. BERTScore
    print("  Computing BERTScore...")
    try:
        P, R, F1_bert = bert_score_fn(responses, correct_answer_texts, lang="en", verbose=False, rescale_with_baseline=True)
        avg_bertscore_p  = round(P.mean().item(), 4)
        avg_bertscore_r  = round(R.mean().item(), 4)
        avg_bertscore_f1 = round(F1_bert.mean().item(), 4)
    except Exception as e:
        print(f"  BERTScore error: {e}")
        avg_bertscore_p = avg_bertscore_r = avg_bertscore_f1 = np.nan

    # 5. Retrieval metrics
    ret_metrics = compute_retrieval_metrics(all_retrieved_indices, df_questions, all_chunks)

    # 6. RAGAS metrics (LLM-based)
    print("  Computing RAGAS metrics (LLM-based)...")
    faith_scores, relevance_scores, ctx_precision_scores, ctx_recall_scores = [], [], [], []
    for i in tqdm(range(len(df_questions)), desc="RAGAS eval", leave=False):
        context_str = "\n".join(all_context_texts[i]) if all_context_texts[i] else ""
        faith_scores.append(ragas_faithfulness_single(responses[i], context_str, groq_client))
        relevance_scores.append(ragas_answer_relevance_single(questions_text[i], responses[i], groq_client))
        ctx_precision_scores.append(ragas_context_precision_single(
            questions_text[i], all_context_texts[i], correct_answer_texts[i], groq_client))
        ctx_recall_scores.append(ragas_context_recall_single(
            questions_text[i], context_str, correct_answer_texts[i], groq_client))
        time.sleep(0.3)

    valid_faith = [s for s in faith_scores if not np.isnan(s)]
    hallucination_rate = round(
        sum(1 for s in valid_faith if s < FAITHFULNESS_THRESHOLD) / len(valid_faith), 4
    ) if valid_faith else np.nan

    # Aggregate all metrics
    metrics = {
        **cls_metrics,
        "Token_F1": avg_token_f1, "Exact_Match": avg_exact_match,
        "ROUGE_1": round(np.mean(rouge1_scores), 4), "ROUGE_L": round(np.mean(rougeL_scores), 4),
        "BERTScore_P": avg_bertscore_p, "BERTScore_R": avg_bertscore_r, "BERTScore_F1": avg_bertscore_f1,
        **ret_metrics,
        "RAGAS_Faithfulness": round(np.nanmean(faith_scores), 4),
        "RAGAS_Hallucination_Rate": hallucination_rate,
        "RAGAS_Answer_Relevance": round(np.nanmean(relevance_scores), 4),
        "RAGAS_Context_Precision": round(np.nanmean(ctx_precision_scores), 4),
        "RAGAS_Context_Recall": round(np.nanmean(ctx_recall_scores), 4),
    }

    config = {
        "SN": excel_row, "RAG_Name": rag_display, "Chunking_Technique": CHUNKING_NAME,
        "Embedding": EMBEDDING_MODEL_NAME, "VectorDB": vectordb_name, "Generator_Model": LLM_MODEL,
    }

    # Write to Excel
    if excel_row:
        write_experiment_to_excel(EXCEL_PATH, excel_row, config, metrics)

    # Store results
    result_key = f"{rag_type}_{vectordb_name.lower()}"
    all_results[result_key] = {"config": config, "metrics": metrics}

    print(f"\n--- {rag_display} + {vectordb_name} RESULTS ---")
    for k, v in metrics.items():
        print(f"  {k:30s}: {v}")

    return metrics


print("Experiment runner and Excel writer defined.")

---

## 11. Phase 1: FAISS VectorDB Experiments

Build a FAISS index and run all 4 RAG architectures.

In [ ]:
# Build FAISS index
print("Building FAISS index...")
faiss_index = faiss.IndexFlatIP(EMBEDDING_DIM)
faiss_index.add(chunk_embeddings)
print(f"FAISS index built: {faiss_index.ntotal:,} vectors, dimension={EMBEDDING_DIM}")

### 11.1 Naive RAG + FAISS (Excel Row 2)

In [ ]:
run_full_experiment("naive", "FAISS", df_sample, faiss_idx=faiss_index)

### 11.2 Sparse RAG + FAISS (Excel Row 7)

In [ ]:
run_full_experiment("sparse", "FAISS", df_sample, faiss_idx=faiss_index)

### 11.3 Hybrid RAG + FAISS (Excel Row 12)

In [ ]:
run_full_experiment("hybrid", "FAISS", df_sample, faiss_idx=faiss_index)

### 11.4 Multi-Hop RAG + FAISS (Excel Row 17)

In [ ]:
run_full_experiment("multihop", "FAISS", df_sample, faiss_idx=faiss_index)

---

## 12. Phase 2: ChromaDB VectorDB Experiments

Build a ChromaDB collection and run all 4 RAG architectures.

In [ ]:
# Build ChromaDB collection
print("Building ChromaDB collection...")
chroma_client = chromadb.Client()
try: chroma_client.delete_collection("medqa_chunks")
except: pass
chroma_collection = chroma_client.create_collection("medqa_chunks", metadata={"hnsw:space": "cosine"})
BATCH_SIZE = 5000
for start in tqdm(range(0, len(all_chunks), BATCH_SIZE), desc="Adding to ChromaDB"):
    end = min(start + BATCH_SIZE, len(all_chunks))
    chroma_collection.add(
        ids=[str(i) for i in range(start, end)],
        embeddings=chunk_embeddings[start:end].tolist(),
        documents=all_chunks[start:end],
        metadatas=[chunk_metadata[i] for i in range(start, end)],
    )
print(f"ChromaDB collection built: {chroma_collection.count():,} chunks")

### 12.1 Naive RAG + ChromaDB (Excel Row 3)

In [ ]:
run_full_experiment("naive", "ChromaDB", df_sample, chroma_col=chroma_collection)

### 12.2 Sparse RAG + ChromaDB (Excel Row 8)

In [ ]:
run_full_experiment("sparse", "ChromaDB", df_sample, chroma_col=chroma_collection)

### 12.3 Hybrid RAG + ChromaDB (Excel Row 13)

In [ ]:
run_full_experiment("hybrid", "ChromaDB", df_sample, chroma_col=chroma_collection)

### 12.4 Multi-Hop RAG + ChromaDB (Excel Row 18)

In [ ]:
run_full_experiment("multihop", "ChromaDB", df_sample, chroma_col=chroma_collection)

---

## 13. Phase 3: Qdrant VectorDB Experiments

Build a Qdrant in-memory collection and run all 4 RAG architectures.

In [ ]:
# Build Qdrant collection (in-memory)
print("Building Qdrant collection...")
qdrant_client_obj = QdrantClient(":memory:")
QDRANT_COLLECTION = "medqa_chunks"
qdrant_client_obj.create_collection(
    collection_name=QDRANT_COLLECTION,
    vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
)
BATCH_SIZE = 5000
for start in tqdm(range(0, len(all_chunks), BATCH_SIZE), desc="Adding to Qdrant"):
    end = min(start + BATCH_SIZE, len(all_chunks))
    points = [
        PointStruct(id=i, vector=chunk_embeddings[i].tolist(),
                    payload={"text": all_chunks[i], "book": chunk_metadata[i]["book_name"]})
        for i in range(start, end)
    ]
    qdrant_client_obj.upsert(collection_name=QDRANT_COLLECTION, points=points)
print(f"Qdrant collection built: {qdrant_client_obj.count(QDRANT_COLLECTION).count:,} points")

### 13.1 Naive RAG + Qdrant (Excel Row 4)

In [ ]:
run_full_experiment("naive", "Qdrant", df_sample, qdrant_cl=qdrant_client_obj, qdrant_col_name=QDRANT_COLLECTION)

### 13.2 Sparse RAG + Qdrant (Excel Row 9)

In [ ]:
run_full_experiment("sparse", "Qdrant", df_sample, qdrant_cl=qdrant_client_obj, qdrant_col_name=QDRANT_COLLECTION)

### 13.3 Hybrid RAG + Qdrant (Excel Row 14)

In [ ]:
run_full_experiment("hybrid", "Qdrant", df_sample, qdrant_cl=qdrant_client_obj, qdrant_col_name=QDRANT_COLLECTION)

### 13.4 Multi-Hop RAG + Qdrant (Excel Row 19)

In [ ]:
run_full_experiment("multihop", "Qdrant", df_sample, qdrant_cl=qdrant_client_obj, qdrant_col_name=QDRANT_COLLECTION)

---

## 14. Phase 4: Pinecone VectorDB Experiments (Cloud)

Build a Pinecone serverless index and run all 4 RAG architectures.  
**Skipped automatically** if `PINECONE_API_KEY` is not set.

In [ ]:
# Build Pinecone index (only if API key is set)
pinecone_index = None
if USE_PINECONE:
    print("Building Pinecone index...")
    from pinecone import Pinecone, ServerlessSpec
    pc = Pinecone(api_key=PINECONE_API_KEY)
    PC_INDEX_NAME = "medqa-experiment"
    if PC_INDEX_NAME not in [idx.name for idx in pc.list_indexes()]:
        pc.create_index(name=PC_INDEX_NAME, dimension=EMBEDDING_DIM, metric="cosine",
                        spec=ServerlessSpec(cloud="aws", region="us-east-1"))
        time.sleep(10)
    pinecone_index = pc.Index(PC_INDEX_NAME)
    BATCH = 100
    for s in tqdm(range(0, len(all_chunks), BATCH), desc="Pinecone upsert"):
        e = min(s + BATCH, len(all_chunks))
        vectors = [(str(i), chunk_embeddings[i].tolist(), {"book": chunk_metadata[i]["book_name"]}) for i in range(s, e)]
        pinecone_index.upsert(vectors=vectors)
    time.sleep(5)
    print(f"Pinecone index built: {pinecone_index.describe_index_stats()['total_vector_count']:,} vectors")
else:
    print("Pinecone SKIPPED (no API key set). Set PINECONE_API_KEY in the API config cell to enable.")

### 14.1 Naive RAG + Pinecone (Excel Row 5)

In [ ]:
if USE_PINECONE:
    run_full_experiment("naive", "Pinecone", df_sample, pinecone_idx=pinecone_index)
else:
    print("Skipped — Pinecone not configured.")

### 14.2 Sparse RAG + Pinecone (Excel Row 10)

In [ ]:
if USE_PINECONE:
    run_full_experiment("sparse", "Pinecone", df_sample, pinecone_idx=pinecone_index)
else:
    print("Skipped — Pinecone not configured.")

### 14.3 Hybrid RAG + Pinecone (Excel Row 15)

In [ ]:
if USE_PINECONE:
    run_full_experiment("hybrid", "Pinecone", df_sample, pinecone_idx=pinecone_index)
else:
    print("Skipped — Pinecone not configured.")

### 14.4 Multi-Hop RAG + Pinecone (Excel Row 20)

In [ ]:
if USE_PINECONE:
    run_full_experiment("multihop", "Pinecone", df_sample, pinecone_idx=pinecone_index)
else:
    print("Skipped — Pinecone not configured.")

---

## 15. Comparative Analysis: All VectorDBs

Side-by-side comparison across all VectorDBs and RAG architectures.

In [ ]:
# Build comparison DataFrame
if all_results:
    comparison_data = []
    for key, result in all_results.items():
        parts = key.split("_", 1)
        rag_type = parts[0]
        vdb = parts[1].replace("chromadb","ChromaDB").replace("faiss","FAISS").replace("qdrant","Qdrant").replace("pinecone","Pinecone")
        rag_display = {"naive": "Naive RAG", "sparse": "Sparse RAG",
                       "hybrid": "Hybrid RAG", "multihop": "Multi-Hop RAG"}.get(rag_type, rag_type)
        row_data = {"RAG": rag_display, "VectorDB": vdb}
        row_data.update(result["metrics"])
        comparison_data.append(row_data)

    comp_df = pd.DataFrame(comparison_data)

    print("="*80)
    print("FULL COMPARISON TABLE")
    print("="*80)
    print(comp_df.to_string(index=False))

    # Save to CSV
    comp_df.to_csv(RESULTS_DIR / "vectordb_comparison_results.csv", index=False)
    print(f"\nResults saved to {RESULTS_DIR / 'vectordb_comparison_results.csv'}")

    # Grouped bar chart
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle("VectorDB Comparison - All RAG Architectures", fontsize=14, fontweight="bold")
    key_metrics = ["Accuracy", "RAGAS_Faithfulness", "RAGAS_Hallucination_Rate",
                   "Recall@5", "MRR", "F1"]
    for ax, metric in zip(axes.flat, key_metrics):
        if metric in comp_df.columns:
            pivot = comp_df.pivot(index="RAG", columns="VectorDB", values=metric)
            pivot.plot(kind="bar", ax=ax, edgecolor="black")
            ax.set_title(metric, fontsize=11); ax.set_ylabel("Score")
            ax.set_ylim(0, 1.05); ax.tick_params(axis="x", rotation=45); ax.legend(fontsize=7)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "plots" / "vectordb_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("\nPlot saved to results/experiments/plots/vectordb_comparison.png")

    # Heatmap
    fig, ax = plt.subplots(figsize=(18, 8))
    metric_cols = [c for c in comp_df.columns if c not in ["RAG", "VectorDB"]]
    heatmap_data = comp_df.set_index(comp_df["RAG"] + " + " + comp_df["VectorDB"])[metric_cols]
    sns.heatmap(heatmap_data.astype(float), annot=True, fmt=".3f", cmap="YlOrRd",
                ax=ax, linewidths=0.5, vmin=0, vmax=1)
    ax.set_title("All Metrics Heatmap - VectorDB Comparison", fontsize=13)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "plots" / "metrics_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No results available for comparison.")

## 16. Final Summary

In [ ]:
print("="*60)
print("VECTORDB COMPARISON EXPERIMENTS - FINAL SUMMARY")
print("="*60)
print(f"\nFixed Configuration:")
print(f"  Chunking:  {CHUNKING_NAME}")
print(f"  Embedding: {EMBEDDING_MODEL_NAME}")
print(f"  LLM:       {LLM_MODEL}")
print(f"  Sample:    {SAMPLE_SIZE} dev questions")

if all_results:
    print(f"\nExperiments completed: {len(all_results)}")
    print(f"Results written to: {EXCEL_PATH}")
    print(f"CSV export: {RESULTS_DIR / 'vectordb_comparison_results.csv'}")

    # Save JSON
    serialisable_results = {}
    for key, val in all_results.items():
        serialisable_results[key] = {
            "config": val["config"],
            "metrics": {k: (float(v) if isinstance(v, (np.floating, float)) and not np.isnan(v) else None)
                       for k, v in val["metrics"].items()},
        }
    with open(RESULTS_DIR / "vectordb_experiment_results.json", "w") as f:
        json.dump(serialisable_results, f, indent=2)
    print(f"JSON export: {RESULTS_DIR / 'vectordb_experiment_results.json'}")

    print("\nAll experiments complete. Check the Excel file for full results.")
else:
    print("\nNo experiments have been run yet.")